# 15. Ders: Yaratıcı Kelimeler (Yılın Şiiri Projesi - Bölüm 1)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

ChatGPT, Gemini veya Claude gibi sistemlerle konuştuğunuzda verdikleri cevapların daha önce bir yerlerde depolandığını ve oradan kopyalanıp size gönderildiğini zannedebilirsiniz. Bu devasa bir yanılgıdır! Yapay zeka sistemleri bir ansiklopedi değil, bir 'olasılık motorudur'. 

Yapay zeka modellerinin temeli "Next Token Prediction" (Sıradaki Kelimeyi Tahmin Etme) teknolojisine dayanır. Siz "Bugün hava çok..." dediğinizde, modelin içindeki devasa istatistik ağı, bu cümleden sonra "güzel", "kötü", "soğuk" kelimelerinin gelme ihtimalini hesaplar. En yüksek ihtimalli kelimeyi seçer, metnin sonuna ekler, sonra o yeni metne bakarak bir sonraki kelimeyi tahmin eder. Yaratıcı yazarlık, hikaye üretimi veya kod yazımı tamamen bu kelime kelime tahmin döngüsünün eseridir. GCA Coding Jam'deki 'Year in Poetry' (Yılın Şiiri) projemizin altyapısını bu derste kuracağız.

## Adım 2: Teknik Terimler ve Sözlük

- Text Generation (Metin Üretimi): Verilen bir başlangıç metnini (istem) yapay zekanın kendi kelimeleriyle devam ettirerek yeni bir içerik oluşturmasıdır.
- Prompt (İstem): Modele verilen başlama komutu veya giriş metnidir (Örn: "Bir varmış bir yokmuş").
- Next Token Prediction (Sıradaki Tokenı Tahmin Etme): GPT (Generative Pre-trained Transformer) tarzı modellerin sıradaki kelimeyi matematiksel olasılığa göre hesaplaması.
- Temperature (Sıcaklık/Yaratıcılık Skoru): Modelin ne kadar güvenli veya ne kadar riskli (çılgın/yaratıcı) tahminler yapacağını kontrol eden ayar (Genellikle 0 ile 1 arasındadır).
- GPT-2: OpenAI tarafından geliştirilen ve bugün kullanılan devasa GPT-4 modellerinin açık kaynaklı ve hafif olan, kendi bilgisayarımızda çalıştırabileceğimiz dedesidir.

## Adım 3: Hugging Face Text-Generation Pipeline İçe Aktarma

Bu görevimiz için pipeline ismini `text-generation` olarak ayarlayacağız ve efsanevi `gpt2` modelini indireceğiz.

In [ ]:
from transformers import pipeline
import logging

# Gereksiz terminal uyarılarını gizlemek için
logging.getLogger("transformers").setLevel(logging.ERROR)

# GPT-2 modelini text-generation göreviyle belleğe indiriyoruz.
# (İlk çalıştırmada model boyutu sebebiyle bir süre bekletebilir)
yazar_model = pipeline("text-generation", model="gpt2")

print("Yapay Zeka Yazarımız (GPT-2) kalemi eline aldı.")

## Adım 4: Başlangıç (Prompt) Cümlesi Verme ve Metin Üretimi

GPT modellerine sadece "Şiir yaz" demek çoğu zaman yeterli olmaz (İlerleyen derslerde Instruction Tuning ile komut almayı öğreneceğiz). GPT-2 gibi ham modellere sadece bir başlangıç cümlesi verilir ve o cümleyi tamamlaması (autocomplete) beklenir.

In [ ]:
# Modele vereceğimiz başlangıç kıvılcımı (Prompt)
baslangic_cumlesi = "In the dark and silent forest, a brave knight"

# max_new_tokens: Modele kendi yazdığı kısımdan sonra en fazla kaç kelime eklemesi gerektiğini söyler.
uretilen_sonuc = yazar_model(baslangic_cumlesi, max_new_tokens=30, pad_token_id=50256)

print("--- YAPAY ZEKANIN YAZDIĞI HİKAYE ---")
# Çıktı yine liste içindeki bir sözlüktür
print(uretilen_sonuc[0]['generated_text'])

## Adım 5: Yaratıcılık Ayarları (Temperature)

Yukarıdaki hücreyi 5 kez çalıştırırsanız, bazen aynı kelimeleri tekrar ettiğini görebilirsiniz. Modelin 'sıradan' olasılıklara değil, az bilinen (sürpriz) kelimelere yönelmesini istiyorsak `temperature` parametresini artırırız.

- Temperature = 0.1 -> Çok sıkıcı, çok garantici, genelde düz ansiklopedi gibi.
- Temperature = 0.7 -> Dengeli bir yaratıcılık (Genelde ideali budur).
- Temperature = 1.5 -> Çok çılgın, mantık hatalarına veya rastgele kelimelere (halüsinasyon) meyilli.

In [ ]:
prompt = "The secret of the universe is"

# Düşük yaratıcılık (Garantici)
sikici_sonuc = yazar_model(prompt, max_new_tokens=20, temperature=0.1, pad_token_id=50256)

# Yüksek yaratıcılık (Çılgın)
cilgin_sonuc = yazar_model(prompt, max_new_tokens=20, temperature=1.5, pad_token_id=50256)

print("Sıkıcı / Garantici Model:")
print(sikici_sonuc[0]['generated_text'])
print("\n" + "-"*40 + "\n")
print("Çılgın / Yaratıcı Model:")
print(cilgin_sonuc[0]['generated_text'])

## Adım 6: Aynı Girdiden Birden Fazla Farklı Çıktı Üretme

Aynı anda 3 farklı şiir alternatifi isteyip içinden en iyisini seçmek isteyebiliriz. Bunun için `num_return_sequences` kullanılır.

In [ ]:
baslangic = "To fly in the sky like a bird, you must"

# num_return_sequences=3 ile aynı cümleden çıkan 3 ayrı olasılık dalını (farklı evrenleri) göreceğiz.
coklu_sonuclar = yazar_model(baslangic, max_new_tokens=15, num_return_sequences=3, temperature=0.8, pad_token_id=50256)

print("--- YAPAY ZEKANIN 3 FARKLI SENARYOSU ---")
for sira, yazi in enumerate(coklu_sonuclar):
    print(f"\nSenaryo {sira + 1}:")
    print(yazi['generated_text'])

## Adım 7: Yılın Şiiri Projesine Giriş Denemesi

Kullanıcının yazdığı belirli kelimeleri (Örn: Yıl 2026, Yaz, Mutluluk) modele bir şablon içinde sunarsak, model onu bir sanat eserine bağlamaya çalışacaktır.

In [ ]:
yil = 2026
kelime = "freedom"

# İstem Mühendisliği (Prompt Engineering) temeli: Modele yön vermek.
yonlendirmeli_prompt = f"In the year {yil}, the world was full of {kelime}. A poem about this time:"

siir_denemesi = yazar_model(yonlendirmeli_prompt, max_new_tokens=40, temperature=0.7, pad_token_id=50256)

print("--- YAPAY ZEKA ŞİİRİ ---")
print(siir_denemesi[0]['generated_text'])

## Adım 8: Dene ve Keşfet - Delirme Noktasını Bulma

Temperature ayarını çok artırdığınızda modelin nasıl mantıksız harf kombinasyonları, uydurma kelimeler veya ardışık anlamsız semboller (halüsinasyon) ürettiğini görmek için Adım 5'teki `temperature=1.5` değerini `2.5` ve `3.0` olarak değiştirip kodları çalıştırın.

## Adım 9: Sıra Sende Görevi - Masal Anlatan Yapay Zeka

Problem: Küçük çocuklar için, kullanıcının belirlediği kahraman ismiyle bir masal başlatan, ancak masalın devamını tamamen yapay zekanın hayal gücüne bırakan bir interaktif masal motoru yazmanız isteniyor.

Gereksinimler:
1. `masal_anlat(kahraman_ismi)` adında bir fonksiyon oluşturun.
2. Fonksiyonun içinde `f-string` kullanarak İngilizce bir giriş promptu hazırlayın (Örn: `"Once upon a time, a little hero named {kahraman_ismi} found a magic sword."`).
3. Bu promptu `yazar_model`'e verin. 
4. Yaratıcılık katsayısını (temperature) 0.8 yapın, en fazla 50 yeni token (`max_new_tokens=50`) istesin.
5. Çıkan hikayeyi `print()` ile ekrana yazdırın.
6. Dışarıda fonksiyonu kendi adınızı vererek (`masal_anlat("Ahmet")`) çağırıp test edin.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 15. Dersi bitirerek, bugün dünyayı kasıp kavuran GPT devriminin ana çalışma prensibini (Next Token Prediction) kendi ellerinizle kodladınız. Yapay zekanın nasıl yaratıcı yazarlık yaptığını anladınız.

Bir sonraki dersimizde 'İstem Mühendisliği' (Prompt Engineering) kavramının derinlerine inip, bu asi yazarın saçmalamasını engelleyerek tam olarak istediğimiz formata uymasını sağlayacağız.